

## Q1. What is DevOps in Databricks?

DevOps combines development and operations practices to build, test, deploy and maintain data engineering applications reliably.

In a Databricks project, DevOps includes:

1. Git for version control.
2. CI/CD for automated testing and deployment.
3. Databricks CLI for workspace operations.
4. Declarative Automation Bundles (formerly Databricks Asset Bundles) for deployment.
5. Unit testing for PySpark transformations.
6. Environment management for Dev, UAT and Production.
7. Monitoring, security and cost optimization.

**Example: E-Commerce Lakehouse**

```text
Developer
    |
    v
GitHub Repository
    |
    v
GitHub Actions – CI
    |
    v
Unit Tests + Bundle Validation
    |
    v
Deploy to DEV
    |
    v
UAT Testing
    |
    v
Production Approval
    |
    v
Deploy to PROD
```

**Interview Answer:**

I use Git for version control, GitHub Actions for CI/CD, PySpark unit tests for validating transformations and Declarative Automation Bundles for deploying Databricks resources across environments.



## Q2. What is the difference between CI, Continuous Delivery and Continuous Deployment?

**Answer:**

| Practice | Meaning | Example |
|---|---|---|
| Continuous Integration | Automatically build and test code changes | Run PySpark tests on pull requests |
| Continuous Delivery | Keep changes deployable; production deployment requires approval | Deploy to UAT and wait for production approval |
| Continuous Deployment | Automatically deploy changes that pass required checks | Deploy every approved main-branch change automatically |

**Example Workflow:**

```text
Developer Push
      |
      v
Run Unit Tests
      |
      v
Validate Bundle
      |
      v
Merge to Main
      |
      v
Deploy to DEV
      |
      v
Deploy to UAT
      |
      v
Production Approval
      |
      v
Deploy to PROD
```

**Interview Answer:**

CI validates code changes. Continuous Delivery ensures the application is ready for deployment, while Continuous Deployment automatically deploys changes after successful checks.

For a production data platform, I would initially prefer Continuous Delivery with a production approval gate.




## Q3. Explain the Git workflow used in a Databricks project.

**Answer:**

Git is used to maintain source code history, collaborate with team members and manage code changes.

**Example Git Workflow:**

```text
main
 |
 +---- feature/bronze-ingestion
 |            |
 |            v
 |       Code Changes
 |            |
 |            v
 |       Unit Testing
 |            |
 |            v
 |       Pull Request
 |
 +---- Code Review
             |
             v
         Merge to Main
             |
             v
         CI Pipeline
```

**Common Git Commands:**

```bash
# Clone repository
git clone https://github.com/username/E-Commerce-Lakehouse.git

# Create feature branch
git checkout -b feature/bronze-ingestion

# Check modified files
git status

# Stage changes
git add .

# Commit changes
git commit -m "Add bronze ingestion logic"

# Push feature branch
git push origin feature/bronze-ingestion

# Switch to main
git checkout main

# Pull latest changes
git pull origin main
```

**Recommended Project Structure:**

```text
E-Commerce-Lakehouse/
│
├── notebooks/
├── src/
│   ├── ingestion/
│   ├── transformations/
│   └── quality/
│
├── tests/
├── resources/
│   └── ecommerce_job.yml
│
├── databricks.yml
├── requirements.txt
├── .gitignore
└── README.md
```

**Important:** Never commit passwords, access tokens, private keys or `.env` files containing credentials. Use approved identity-based authentication or Databricks secrets.




## Q4. What is Databricks CLI? How do you authenticate?

**Answer:**

Databricks CLI is a command-line tool used to interact with Databricks workspaces and accounts.

It allows us to manage jobs, clusters, workspace files, Unity Catalog resources and deployment bundles.

**Install on Windows:**

```powershell
winget install Databricks.DatabricksCLI
```

**Check Version:**

```powershell
databricks -v
```

**Authenticate:**

```powershell
databricks auth login --host https://YOUR-WORKSPACE-URL
```

**List Authentication Profiles:**

```powershell
databricks auth profiles
```

**Check Authentication Configuration:**

```powershell
databricks auth env --profile DEFAULT
```

**Important Commands:**

| Command | Purpose |
|---|---|
| `databricks -h` | Display help |
| `databricks jobs list` | List jobs |
| `databricks clusters list` | List clusters |
| `databricks catalogs list` | List catalogs |
| `databricks schemas list workspace` | List schemas |
| `databricks jobs get JOB_ID` | Get job details |
| `databricks workspace ls /Workspace` | List workspace objects |
| `databricks secrets list-scopes` | List secret scope names |

**List Files in Your Unity Catalog Volume:**

```powershell
databricks fs ls dbfs:/Volumes/workspace/default/data_volume/
```

**Important:** The CLI filesystem commands require `dbfs:/Volumes/...` for Unity Catalog Volumes. This is different from the `/Volumes/...` path used in Spark and Python code.

For local development, OAuth user login is appropriate. For CI/CD, use a service principal with OAuth machine-to-machine authentication or workload identity federation.



## Q5. What are Declarative Automation Bundles (DAB)?

**Answer:**

Declarative Automation Bundles, formerly called Databricks Asset Bundles, package Databricks source code and resource definitions into a deployable project.

They help manage:

- Lakeflow Jobs.
- Lakeflow Declarative Pipelines.
- Notebooks and Python files.
- SQL files.
- Environment-specific configurations.
- Other supported Databricks resources.

**Example Project Structure:**

```text
E-Commerce-Lakehouse/
│
├── databricks.yml
│
├── resources/
│   ├── ecommerce_job.yml
│   └── ecommerce_pipeline.yml
│
├── src/
│   ├── bronze_ingestion.py
│   ├── silver_transformation.py
│   └── gold_aggregation.py
│
├── tests/
│   ├── test_ingestion.py
│   └── test_transformations.py
│
├── requirements.txt
└── README.md
```

**Interview Answer:**

I use Declarative Automation Bundles to define Databricks resources as code, maintain them in Git and deploy the same project to development, UAT and production using different deployment targets.



## Q6. What is databricks.yml?

**Answer:**

`databricks.yml` is the main configuration file at the root of a Declarative Automation Bundle.

It defines the bundle name, resource files, variables and deployment targets.

**Example:**

```yaml
bundle:
  name: ecommerce_lakehouse

include:
  - resources/*.yml

variables:
  catalog:
    description: Target catalog
    default: workspace

  schema:
    description: Target schema
    default: default

targets:

  dev:
    mode: development
    default: true

    workspace:
      host: https://YOUR-DEV-WORKSPACE-URL

  uat:
    mode: production

    workspace:
      host: https://YOUR-UAT-WORKSPACE-URL

  prod:
    mode: production

    workspace:
      host: https://YOUR-PROD-WORKSPACE-URL
```

**Explanation:**

- `bundle`: Defines the project name.
- `include`: Includes resource YAML files.
- `variables`: Defines reusable configuration values.
- `targets`: Defines deployment environments.
- `workspace.host`: Specifies the target workspace.

**Important:** Development mode and production mode have different deployment behaviors. Separate workspaces and catalogs provide stronger environment isolation.


## Q7. What are Bundle Variables and Deployment Targets?

**Answer:**

Bundle variables allow us to reuse configuration values instead of hardcoding environment-specific settings.

Deployment targets define where the bundle should be deployed.

**Example:**

```yaml
variables:
  catalog:
    description: Target catalog
    default: workspace

targets:

  dev:
    default: true

    variables:
      catalog: ecommerce_dev

  uat:
    variables:
      catalog: ecommerce_uat

  prod:
    variables:
      catalog: ecommerce_prod
```

**Reference the Variable:**

```yaml
catalog: ${var.catalog}
```

**Environment Comparison:**

| Environment | Catalog |
|---|---|
| DEV | ecommerce_dev |
| UAT | ecommerce_uat |
| PROD | ecommerce_prod |

**Interview Answer:**

I use deployment targets and bundle variables to deploy the same application code into different environments without changing the source code manually.



## Q8. What are the important Declarative Automation Bundle commands?

**Answer:**

| Command | Purpose |
|---|---|
| `databricks bundle init` | Initialize a bundle |
| `databricks bundle validate` | Validate configuration |
| `databricks bundle deploy -t dev` | Deploy to development |
| `databricks bundle deploy -t uat` | Deploy to UAT |
| `databricks bundle deploy -t prod` | Deploy to production |
| `databricks bundle run -t dev ecommerce_job` | Run a deployed job |
| `databricks bundle summary -t dev` | Display deployed resources |
| `databricks bundle destroy -t dev` | Remove bundle-managed resources |

**Complete Example:**

```powershell
# Initialize bundle
databricks bundle init

# Validate configuration
databricks bundle validate -t dev

# Deploy to development
databricks bundle deploy -t dev

# Display deployment summary
databricks bundle summary -t dev

# Run deployed job
databricks bundle run -t dev ecommerce_job
```

**Important:** `bundle destroy` is destructive. Never use it casually in shared or production environments.




## Q9. How do you deploy a Lakeflow Job using Declarative Automation Bundles?

**Answer:**

I define the Job in a resource YAML file and deploy it through the bundle.

**Example: `resources/ecommerce_job.yml`**

```yaml
resources:
  jobs:
    ecommerce_bronze_job:

      name: ecommerce_bronze_job

      tasks:

        - task_key: bronze_ingestion

          notebook_task:
            notebook_path: ../src/bronze_ingestion.py
```

**Deployment Commands:**

```powershell
# Validate
databricks bundle validate -t dev

# Deploy
databricks bundle deploy -t dev

# Run
databricks bundle run -t dev ecommerce_bronze_job
```

**Explanation:**

1. Define the Job in YAML.
2. Include the resource file in `databricks.yml`.
3. Validate the bundle.
4. Deploy it to the development workspace.
5. Run the deployed Job.
6. Verify the execution in Databricks.

This is a simplified resource definition. Compute, libraries, permissions and other settings must be configured according to the actual task and workspace requirements.



## Q10. What is GitHub Actions, and how do you implement CI?

**Answer:**

GitHub Actions is a CI/CD automation platform used to execute workflows when code is pushed or pull requests are created.

It can automate:

- Dependency installation.
- PySpark unit testing.
- Bundle validation.
- Deployment.
- Notifications.

**Example CI Workflow: `.github/workflows/ci.yml`**

```yaml
name: E-Commerce Lakehouse CI

on:
  pull_request:
    branches: [main]

  push:
    branches: [main]

jobs:

  test-and-validate:

    runs-on: ubuntu-latest

    steps:

      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install pytest pyspark

      - name: Run unit tests
        run: |
          pytest -v

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate bundle
        run: |
          databricks bundle validate -t dev
```

**Explanation:**

1. GitHub checks out the repository.
2. Python is configured.
3. Required dependencies are installed.
4. PySpark unit tests are executed.
5. Databricks CLI is installed.
6. Bundle configuration is validated.

**Important:** This is a CI example. It does not deploy to production. Add a separate deployment job with appropriate authentication, environment protection and approval controls.

For automated Databricks authentication, prefer workload identity federation where supported. It avoids storing long-lived client secrets in GitHub.



## Q11. How do you perform Unit Testing in PySpark?

**Answer:**

Unit testing validates individual transformation functions using small test datasets before running the complete pipeline.

For the E-Commerce project, I test:

- Null-value handling.
- Duplicate removal.
- Invalid order amounts.
- Data type conversions.
- Revenue calculations.

**File: `src/transformations.py`**

```python
from pyspark.sql import functions as F


def clean_orders(df):

    return (
        df
        .filter(
            F.col("order_id").isNotNull()
        )
        .filter(
            F.col("total_amount") > 0
        )
        .dropDuplicates(
            ["order_id"]
        )
    )


def calculate_revenue(df):

    return (
        df
        .groupBy("order_date")
        .agg(
            F.sum("total_amount")
            .alias("total_revenue")
        )
    )
```

**File: `tests/test_transformations.py`**

```python
import pytest

from pyspark.sql import SparkSession

from src.transformations import clean_orders


@pytest.fixture(scope="session")
def spark():

    session = (
        SparkSession.builder
        .master("local[2]")
        .appName("ECommerceUnitTests")
        .getOrCreate()
    )

    yield session

    session.stop()


def test_clean_orders(spark):

    data = [
        (1, 100.0),
        (1, 100.0),
        (2, -50.0),
        (None, 200.0)
    ]

    columns = [
        "order_id",
        "total_amount"
    ]

    df = spark.createDataFrame(
        data,
        columns
    )

    result = clean_orders(df)

    assert result.count() == 1

    assert (
        result.first()["order_id"] == 1
    )
```

**Run Tests:**

```bash
pip install pytest pyspark

pytest -v
```

**Expected Result:**

```text
test_clean_orders PASSED

1 passed
```

**Important:** This tests local Spark transformation logic. It does not prove that Unity Catalog permissions, Databricks compute, external connections or production pipeline execution will work. Those require integration testing.




## Q12. How do you optimize Databricks compute costs?

**Answer:**

Cost optimization means reducing unnecessary compute, storage and query expenses without compromising required performance.

**Important Techniques:**

| Technique | Purpose | E-Commerce Example |
|---|---|---|
| Autoscaling | Adjust compute based on workload | Scale workers during large ingestion |
| Auto termination | Stop idle compute | Stop unused development compute |
| Serverless | Reduce infrastructure management | Run supported workloads on serverless |
| Photon | Accelerate supported workloads | Improve SQL aggregation performance |
| Incremental processing | Process only new or changed data | Process newly arrived orders |
| Partition pruning | Read only required partitions | Query selected order dates |
| Broadcast join | Reduce shuffle for small tables | Join small product dimension |
| File compaction | Reduce small-file overhead | Optimize Delta tables |
| Right-sized warehouse | Match compute to query workload | Avoid oversized SQL warehouses |

**SQL Example – Query Only Required Dates:**

```sql
SELECT
    order_date,
    SUM(total_amount) AS total_revenue
FROM workspace.ecommerce_gold.daily_sales
WHERE order_date >= '2026-11-01'
  AND order_date < '2026-12-01'
GROUP BY order_date;
```

**PySpark Example – Broadcast Join:**

```python
from pyspark.sql import functions as F

result_df = (
    orders_df
    .join(
        F.broadcast(products_df),
        "product_id",
        "inner"
    )
)
```

**Incremental Processing Example:**

```python
new_orders_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .load(
        "/Volumes/workspace/default/data_volume/ecommerce_lakehouse/landing/orders/"
    )
)
```

**Interview Answer:**

I optimize costs using incremental processing, suitable compute sizing, autoscaling, auto termination, efficient joins, SQL query optimization and monitoring. I compare execution time and compute consumption before and after optimization.



## Q13. What is the difference between Terraform, Ansible and Declarative Automation Bundles?

**Answer:**

These tools solve different problems in DevOps.

| Tool | Purpose | Example |
|---|---|---|
| Terraform | Infrastructure provisioning | Provision cloud and Databricks resources |
| Ansible | Configuration management and automation | Configure servers |
| Declarative Automation Bundles | Deploy Databricks resources | Deploy Lakeflow Jobs |
| GitHub Actions | CI/CD automation | Run tests and deploy bundles |
| Jenkins | CI/CD automation server | Enterprise deployment pipeline |

**Example:**

```text
Terraform
    |
    v
Provision Infrastructure
    |
    v
Databricks Workspace
    |
    v
Declarative Automation Bundles
    |
    v
Deploy Jobs and Pipelines
    |
    v
GitHub Actions
    |
    v
Automated CI/CD
```

**Interview Answer:**

I use Terraform for infrastructure provisioning, Declarative Automation Bundles for Databricks workload deployment and GitHub Actions or Jenkins to automate testing and deployment.




## Q14. How do you secure credentials and manage environments in Databricks DevOps?

**Answer:**

I follow these practices:

1. Never store credentials in Git.
2. Use Databricks secrets or an approved external secret manager.
3. Use service principals for automated deployments.
4. Prefer workload identity federation where supported.
5. Apply least-privilege permissions.
6. Maintain separate development, UAT and production environments.
7. Use environment-specific configuration.
8. Protect production deployment with approvals.

**Example: Access a Secret in a Databricks Notebook**

```python
username = dbutils.secrets.get(
    scope="ecommerce_scope",
    key="database_username"
)

password = dbutils.secrets.get(
    scope="ecommerce_scope",
    key="database_password"
)
```

**Important:** Do not print the secret values or include them in error messages.

**Environment Example:**

| Environment | Catalog |
|---|---|
| DEV | ecommerce_dev |
| UAT | ecommerce_uat |
| PROD | ecommerce_prod |



## Q15. How do you deploy and roll back an E-Commerce Lakehouse project?

**Answer:**

I follow a controlled deployment process.

**Deployment Workflow:**

```text
Developer Changes
       |
       v
Git Feature Branch
       |
       v
Pull Request
       |
       v
Unit Tests
       |
       v
Bundle Validation
       |
       v
Deploy DEV
       |
       v
Integration Tests in UAT
       |
       v
Production Approval
       |
       v
Deploy PROD
       |
       v
Monitor Jobs and Pipelines
```

**Example Deployment Commands:**

```powershell
# Validate development
databricks bundle validate -t dev

# Deploy development
databricks bundle deploy -t dev

# Deploy UAT
databricks bundle deploy -t uat

# Deploy production
databricks bundle deploy -t prod
```

**If Production Deployment Fails:**

1. Identify the failed deployment and affected resources.
2. Stop further promotion.
3. Inspect the deployment logs.
4. Restore the previous known-good code and resource configuration through the deployment process.
5. Validate Jobs and Pipelines.
6. Check data correctness.
7. Resume processing after verification.

**SQL Example – Validate Gold Data:**

```sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(total_amount) AS total_revenue
FROM workspace.ecommerce_gold.daily_sales;
```

**Important:**

Reverting Git code does not automatically undo data already written to Delta tables. If data was corrupted, use the appropriate Delta recovery, restore or backfill strategy after assessing downstream impact.

**Interview Answer:**

I deploy through separate development, UAT and production targets. Before production, I run unit and integration tests and require approval. If a deployment fails, I restore the previous known-good release and separately verify whether any data recovery is required.